# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [1]:
import os
import time
from rtm_pymmcore.data_structures import Channel, StimTreatment
import rtm_pymmcore.utils as utils
from pprint import pprint
import pandas as pd

### Experimental Settings

In [2]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

# for Jungfrau Microscope enable here:
# from rtm_pymmcore.microscope.Jungfrau import Jungfrau

# mic = Jungfrau()
# mic.mmc.setChannelGroup("TTL_ERK")

from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

In [ ]:
## Configuration options
FIRST_FRAME_STIMULATION = 2
# N_FRAMES = 340
N_FRAMES_PHASE_1 = 6
N_FRAMES_PHASE_2 = 7


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 10  # time in seconds between frames
TIME_PER_FOV = 3.75  # time in seconds per fov

ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)


## Storage path for the experiment
base_path = "E:\\Alex"
experiment_name = "Tes5"
path = os.path.join(base_path, experiment_name)


# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=200))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="Red", exposure=300)
optocheck_timepoints = (3,)  # timepoints in frames when optocheck is done

channel_optocheck = Channel(name="mCitrine", exposure=600)
optocheck_timepoints = (4,)

condition = ["test"]

# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps_phase_1 = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_timestep": (2, 3),
        "stim_exposure_list": (100,) * 90,
    }
]

stim_exposures_timesteps_phase_2 = [
    {
        "ramp_pattern_name": "Sustained",
        "stim_timestep": (2, 3),
        "stim_exposure_list": (100,) * 90,
    }
]


for stim_exposures_timesteps in [
    stim_exposures_timesteps_phase_1,
    stim_exposures_timesteps_phase_2,
]:
    utils.fix_tuples_in_stim_exposure_list(stim_exposures_timesteps)
    utils.add_stim_parameters_to_stim_exposures_timesteps(stim_exposures_timesteps)
    utils.print_stim_exposures_timesteps(stim_exposures_timesteps)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=35)
optocheck = OptoCheckFE(used_mask="labels")


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Sustained
100 at 2
100 at 3

Pattern Name:  Sustained
100 at 2
100 at 3

Directory E:\Alex\Tes5\raw already exists
Directory E:\Alex\Tes5\tracks already exists
Directory E:\Alex\Tes5\stim_mask already exists
Directory E:\Alex\Tes5\stim already exists
Directory E:\Alex\Tes5\particles already exists
Directory E:\Alex\Tes5\labels_ring already exists
Directory E:\Alex\Tes5\labels already exists
Directory E:\Alex\Tes5\optocheck already exists


### GUI - Napari Micromanager

#### Load GUI

In [4]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

### Use FOVs to generate dataframe for acquisition

In [5]:
fovs = utils.generate_fov_objects(mic, viewer=viewer)
fovs

df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=0,
    condition=condition,
)
df_acquire = utils.apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_exposures_timesteps_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

Total Experiment Time: 0.013888888888888888h
Doing 2 experiment per stim condition


c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\utils.py:103: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00000,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00000,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,1,10.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00001,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,1,10.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00001,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,2,20.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00002,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
5,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,2,20.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00002,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
6,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,3,30.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00003,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
7,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,3,30.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00003,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
8,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,4,40.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_00_00004,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
9,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,4,40.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_00_00004,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False


In [6]:
df_acquire_2 = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=1,
    condition=condition,
)
df_acquire_2 = utils.apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_exposures_timesteps_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

Total Experiment Time: 0.016666666666666666h
Doing 2 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_timestep,stim_exposure_list,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00000,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00000,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,1,10.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00001,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
3,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,1,10.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00001,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
4,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,2,20.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00002,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
5,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,2,20.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00002,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
6,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,3,30.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00003,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
7,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,3,30.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00003,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,100.0,True
8,<rtm_pymmcore.data_structures.Fov object at 0x...,0,44054.1,-21951.4,0,4,40.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_01_00004,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False
9,<rtm_pymmcore.data_structures.Fov object at 0x...,1,44054.1,-22531.4,1,4,40.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_01_00004,test,...,Sustained,"(2, 3)","(100, 100, 100, 100, 100, 100, 100, 100, 100, ...",10,Red,WF_TTL,LedDMD,Cyan_Level,0.0,False


### Run experiment

In [7]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

mic.run_experiment(df_acquire)

In [8]:
mic.run_experiment(df_acquire_2)

In [9]:
mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [18]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()